# Inference & Top-Loss Analysis

> **What you will learn:**  
> Load a trained checkpoint, run inference on the validation set, and surface the worst-predicted patches for error analysis.

---

| Step | What happens |
|------|--------------|
| 0 | Setup — paths, device, config |
| 1 | Load the model from a checkpoint |
| 2 | Build the validation dataset (no rebuild needed) |
| 3 | Visualise predictions on random val samples |
| 4 | Find the operating threshold on the val set |
| 5 | Plot top-loss patches (worst predictions) |
| 6 | Full-image inference on an unseen image |

---
## Step 0 — Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader

from segmentation_tools_pt.core      import get_device
from segmentation_tools_pt.model     import get_model
from segmentation_tools_pt.dataset   import SegmentationDataset
from segmentation_tools_pt.losses    import SegmentationLoss
from segmentation_tools_pt.inference import predict, decode, find_operating_threshold
from segmentation_tools_pt.visualize import show_pred, show_batch, show_top_losses

print(f'torch {torch.__version__} | cuda {torch.cuda.is_available()}')

In [ ]:
# ── Edit these paths to match your project ────────────────────────────────
CHECKPOINT  = Path('../checkpoints/best.pth')          # or periodic_ep*.pth
PATCH_DIR   = Path('/path/to/patches')                 # data.patch_dir from train.yaml
ARCH        = 'mobilenet_v2'                           # model.arch
IN_CHANNELS = 3                                        # model.in_channels (3 for single_repeat_3)
NUM_CLASSES = 1                                        # train.num_classes
THRESHOLD   = 0.5                                      # starting decision threshold
# ──────────────────────────────────────────────────────────────────────────

DEVICE = get_device()
print(f'device: {DEVICE}')

---
## Step 1 — Load the model from a checkpoint

No `pretrained=True` needed — the checkpoint already contains the fully trained weights.

In [ ]:
model = get_model(arch=ARCH, in_channels=IN_CHANNELS, num_classes=NUM_CLASSES)

ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.to(DEVICE)
model.eval()

print(f'Loaded checkpoint from epoch {ckpt["epoch"] + 1}')
print(f'Best score in checkpoint : {ckpt["best_score"]:.4f}')

---
## Step 2 — Build the validation dataset

Reads from the existing patch cache — no rebuild, no training.

In [ ]:
# For data_mode: single_repeat_3 — reads 1 source PNG and repeats it 3x
val_ds = SegmentationDataset(
    root=str(PATCH_DIR),
    split='val',
    num_classes=NUM_CLASSES,
    n_source_channels=1,
    channel_names=['ch0', 'ch0', 'ch0'],   # repeat gray -> 3-channel tensor
)

print(f'Val samples : {len(val_ds)}')
s = val_ds[0]
print(f'image shape : {s["image"].shape}  mask shape : {s["mask"].shape}')

---
## Step 3 — Visualise predictions on random val samples

Colour code:
- **Green** = GT only (model missed it — false negative)
- **Red** = prediction only (false alarm — false positive)
- **Magenta** = both agree (true positive)

In [ ]:
fig = show_batch(model, val_ds, DEVICE, n=8, threshold=THRESHOLD)
plt.suptitle('Random val samples — GT vs Prediction', y=1.01)
plt.show()

---
## Step 4 — Find the operating threshold

Scans thresholds 0.05 → 0.90 and returns the lowest one that still achieves `target_recall`.

In [ ]:
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

result = find_operating_threshold(
    model, val_loader, DEVICE,
    target_recall=0.95,
)

THRESHOLD = result['recommended_threshold']
print(f"Recommended threshold : {THRESHOLD:.2f}")
print(f"Recall at that point  : {result['recall_at_threshold']:.4f}")

In [ ]:
sweep      = result['sweep']
thresholds = [r['threshold'] for r in sweep]
recalls    = [r['recall']    for r in sweep]
precisions = [r['precision'] for r in sweep]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(thresholds, recalls,    label='recall')
ax.plot(thresholds, precisions, label='precision')
ax.axvline(THRESHOLD, color='k', linestyle='--', label=f'recommended ({THRESHOLD:.2f})')
ax.set_xlabel('threshold'); ax.set_ylabel('score')
ax.legend(); ax.set_title('Threshold sweep on val set')
plt.tight_layout()
plt.show()

---
## Step 5 — Top-loss patches (worst predictions)

These are the patches where the model is most confused. Useful for identifying:
- Annotation errors in the ground truth
- Edge cases the model has not seen enough of
- Patches that need re-annotation or augmentation

In [ ]:
criterion = SegmentationLoss(num_classes=NUM_CLASSES)

fig = show_top_losses(model, val_ds, criterion, DEVICE, k=12, threshold=THRESHOLD)
plt.suptitle('Top-loss patches — worst predictions on val set', y=1.01)
plt.show()

---
## Step 6 — Full-image inference on an unseen image

For images that were never patched (true test data).  
Works on any image size — no retraining needed.

In [ ]:
# Point this at any grayscale .png you want to inspect
IMAGE_PATH = Path('/path/to/test_image.png')

# Load as grayscale, normalise to [0, 1], repeat to 3 channels (single_repeat_3)
img_np = np.array(Image.open(IMAGE_PATH).convert('L'), dtype=np.float32) / 255.0
img_t  = torch.tensor(img_np).unsqueeze(0).repeat(3, 1, 1)  # (3, H, W)

probs = predict(model, img_t, DEVICE)           # (1, H, W) probabilities
mask  = decode(probs, threshold=THRESHOLD)      # (H, W)    binary mask

print(f'Image size : {img_np.shape}')
print(f'Defect px  : {mask.sum().item()} / {mask.numel()} ({100*mask.float().mean():.2f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img_np,                         cmap='gray'            ); axes[0].set_title('Input image')
axes[1].imshow(probs[0].cpu().numpy(),         cmap='hot', vmin=0, vmax=1); axes[1].set_title('Probability map')
axes[2].imshow(mask.cpu().numpy(),             cmap='gray'            ); axes[2].set_title(f'Binary mask (thr={THRESHOLD:.2f})')
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Overlay prediction on image (no GT available for test images, pass a blank mask)
blank_gt = torch.zeros_like(mask)
show_pred(img_t, blank_gt, probs, threshold=THRESHOLD)
plt.title('Prediction overlay (red = predicted defect, no GT available)')
plt.show()